# Content Publish Workflow

1. **Add an entry** — pick a type, fill the form, click **Add / Update**.
2. **Curate Recent News** — pick up to 5 entries to feature on the About (index) page.
3. **Build** — regenerate the site bundle.
4. **Publish** — commit (and optionally push).

Entries map to:
- `publication` → About Recent News + Research > Publications
- `presentation` → About Recent News + News (Workshops/Demo also on Playground)
- `project` → About Recent News + Playground
- `news` → About Recent News + News (Others)
- `blog` → Blog only

Required: `date` and `title`. Everything else can be empty.

If widgets don't render, run `pip install ipywidgets` and reload.

In [20]:
from pathlib import Path
import json, re, subprocess, sys, datetime as dt
import ipywidgets as widgets
from IPython.display import display, clear_output

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    while cur != cur.parent:
        if (cur / 'scripts' / 'build_content.py').exists():
            return cur
        cur = cur.parent
    raise RuntimeError('Could not find repo root')

REPO_ROOT = find_repo_root(Path.cwd())
CONTENT_DIR = REPO_ROOT / 'content'
BLOG_DIR = CONTENT_DIR / 'blog'
ENTRIES_PATH = CONTENT_DIR / 'entries.json'
HOME_PATH = CONTENT_DIR / 'home.json'

PUB_TYPES = ['Books', 'Chapters', 'Refereed Journal Articles', 'Refereed Conference Proceedings']
PRES_TYPES = ['Keynotes', 'Invited Talks', 'Refereed Presentations', 'Refereed Posters',
              'Non-Refereed Presentations', 'Non-Refereed Panels', 'Workshops', 'Demo']
PROJECT_MODES = ['external', 'embedded']

ME_DEFAULT = 'Shin, R.'
print('Repo root:', REPO_ROOT)

Repo root: /Users/rosalyn/GitHub/hyunjoors.github.io


## 1. Add or update an entry

In [26]:
type_picker = widgets.RadioButtons(
    options=['publication', 'presentation', 'project', 'news', 'blog'],
    description='Type:',
    style={'description_width': 'initial'},
)
form_box = widgets.VBox()
authors_box = widgets.VBox()
fields = {}
author_rows = []  # list of (name_widget, me_checkbox, row_box)

LW = widgets.Layout(width='620px')
SW = {'description_width': '130px'}

def W_text(label, placeholder=''):
    return widgets.Text(description=label, placeholder=placeholder, layout=LW, style=SW)

def W_area(label, placeholder='', height='100px'):
    return widgets.Textarea(description=label, placeholder=placeholder,
                             layout=widgets.Layout(width='620px', height=height), style=SW)

def W_drop(label, options, value=None):
    return widgets.Dropdown(description=label, options=options,
                            value=value if value is not None else options[0],
                            layout=LW, style=SW)

def W_date(label):
    return widgets.DatePicker(description=label, layout=LW, style=SW)

# --- authors widget ---
def make_author_row(name='', me=False):
    nm = widgets.Text(value=name, placeholder='Author name', layout=widgets.Layout(width='420px'))
    me_cb = widgets.Checkbox(value=me, description='me', indent=False, layout=widgets.Layout(width='80px'))
    rm = widgets.Button(description='✕', layout=widgets.Layout(width='40px'), button_style='')
    row = widgets.HBox([nm, me_cb, rm])
    def on_rm(_):
        author_rows[:] = [r for r in author_rows if r[2] is not row]
        refresh_authors()
    rm.on_click(on_rm)
    return (nm, me_cb, row)

def refresh_authors():
    add_btn = widgets.Button(description='+ Add author', layout=widgets.Layout(width='150px'))
    def on_add(_):
        author_rows.append(make_author_row())
        refresh_authors()
    add_btn.on_click(on_add)
    label = widgets.HTML("<b>Authors</b> (in order, check 'me' for yours)")
    authors_box.children = [label] + [r[2] for r in author_rows] + [add_btn]

# --- form builder ---
def build_form(kind):
    global fields, author_rows
    author_rows = []
    if kind == 'blog':
        fields = {
            'id': W_text('id', 'my-post-slug'),
            'title': W_text('title', 'Post title'),
            'date': W_date('date'),
            'tag': W_text('tag', 'Research / Reflection / ...'),
            'excerpt': W_area('excerpt', 'One-sentence teaser', '60px'),
            'readTime': W_text('readTime', '6 min'),
            'body': W_area('body', 'Markdown post body...', '320px'),
        }
        form_box.children = list(fields.values())
        authors_box.children = []
        return

    common = {
        'id': W_text('id', f'{kind}-my-slug'),
        'date': W_date('date'),
        'title': W_text('title', 'Title'),
        'keywords': W_text('keywords', 'comma,separated,keywords'),
        'description': W_area('description', 'Optional description', '100px'),
    }

    if kind == 'publication':
        fields = {**common,
            'pubType': W_drop('pubType', PUB_TYPES, 'Refereed Conference Proceedings'),
            'venue': W_text('venue', 'Journal / Conference'),
            'pdfUrl': W_text('pdfUrl', 'URL to PDF'),
        }
    elif kind == 'presentation':
        fields = {**common,
            'presType': W_drop('presType', PRES_TYPES, 'Refereed Presentations'),
            'venue': W_text('venue', 'Conference / Lab / Class'),
            'attachmentUrl': W_text('attachmentUrl', 'URL to image or PDF'),
        }
    elif kind == 'project':
        fields = {**common,
            'mode': W_drop('mode', PROJECT_MODES, 'external'),
            'url': W_text('url', 'External URL or projects/.../index.html'),
        }
    elif kind == 'news':
        fields = {**common,
            'url': W_text('url', 'Optional URL'),
        }

    author_rows.append(make_author_row(ME_DEFAULT, me=True))
    form_box.children = list(fields.values())
    refresh_authors()

def on_type_change(_):
    build_form(type_picker.value)

type_picker.observe(on_type_change, names='value')
build_form(type_picker.value)

# --- load existing entry ---
def parse_blog_md(path):
    text = path.read_text(encoding='utf-8')
    lines = text.splitlines()
    if not lines or lines[0].strip() != '---':
        return None
    end = next((i for i in range(1, len(lines)) if lines[i].strip() == '---'), None)
    if end is None:
        return None
    fm = {}
    for line in lines[1:end]:
        if ':' in line:
            k, v = line.split(':', 1)
            fm[k.strip()] = v.strip()
    fm['body'] = '\n'.join(lines[end + 1:]).strip()
    fm['_path'] = path
    return fm

def list_existing_ids():
    out = []
    if ENTRIES_PATH.exists():
        data = json.loads(ENTRIES_PATH.read_text(encoding='utf-8'))
        for e in data.get('entries', []):
            out.append((f"[{e['type']}] {e['id']} — {e.get('title','')[:60]}", ('entry', e['id'])))
    if BLOG_DIR.exists():
        for p in sorted(BLOG_DIR.glob('*.md')):
            fm = parse_blog_md(p)
            if fm and fm.get('id'):
                out.append((f"[blog] {fm['id']} — {fm.get('title','')[:60]}", ('blog', fm['id'])))
    return out

load_picker = widgets.Dropdown(
    options=[('— pick to load —', None)] + list_existing_ids(),
    description='Load existing:',
    layout=widgets.Layout(width='800px'),
    style={'description_width': '120px'},
)
load_btn = widgets.Button(description='Load into form', button_style='info')
refresh_load_btn = widgets.Button(description='Refresh list', button_style='')
clear_btn = widgets.Button(description='Clear form (new entry)', button_style='')
delete_btn = widgets.Button(description='Delete selected', button_style='danger')
delete_confirm = widgets.Checkbox(value=False, description='Confirm delete', indent=False)
load_out = widgets.Output()

def set_field(key, value):
    if key not in fields or value is None:
        return
    w = fields[key]
    if isinstance(w, widgets.DatePicker):
        if isinstance(value, str) and value:
            try:
                w.value = dt.date.fromisoformat(value)
            except ValueError:
                pass
    else:
        w.value = value if value is not None else ''

def load_entry_by_id(entry_id):
    data = json.loads(ENTRIES_PATH.read_text(encoding='utf-8'))
    entry = next((e for e in data['entries'] if e['id'] == entry_id), None)
    if not entry:
        raise ValueError(f"Entry '{entry_id}' not found")
    type_picker.value = entry['type']  # triggers build_form
    set_field('id', entry.get('id', ''))
    set_field('date', entry.get('date'))
    set_field('title', entry.get('title', ''))
    set_field('keywords', ', '.join(entry.get('keywords') or []))
    set_field('description', entry.get('description', ''))
    if entry['type'] == 'publication':
        set_field('pubType', entry.get('pubType') or PUB_TYPES[-1])
        set_field('venue', entry.get('venue', ''))
        set_field('pdfUrl', entry.get('pdfUrl', ''))
    elif entry['type'] == 'presentation':
        set_field('presType', entry.get('presType') or PRES_TYPES[2])
        set_field('venue', entry.get('venue', ''))
        set_field('attachmentUrl', entry.get('attachmentUrl', ''))
    elif entry['type'] == 'project':
        set_field('mode', entry.get('mode') or 'external')
        set_field('url', entry.get('url', ''))
    elif entry['type'] == 'news':
        set_field('url', entry.get('url', ''))
    # rebuild authors
    author_rows.clear()
    for a in (entry.get('authors') or []):
        author_rows.append(make_author_row(a.get('name', ''), bool(a.get('me'))))
    if not author_rows and entry['type'] != 'news':
        author_rows.append(make_author_row(ME_DEFAULT, me=True))
    refresh_authors()

def load_blog_by_id(post_id):
    target = None
    for p in BLOG_DIR.glob('*.md'):
        fm = parse_blog_md(p)
        if fm and fm.get('id') == post_id:
            target = fm
            break
    if not target:
        raise ValueError(f"Blog '{post_id}' not found")
    type_picker.value = 'blog'
    set_field('id', target.get('id', ''))
    set_field('date', target.get('date'))
    set_field('title', target.get('title', ''))
    set_field('tag', target.get('tag', ''))
    set_field('excerpt', target.get('excerpt', ''))
    set_field('readTime', target.get('readTime', ''))
    set_field('body', target.get('body', ''))

def on_load(_):
    with load_out:
        clear_output()
        val = load_picker.value
        if not val:
            print('Pick an entry first.')
            return
        kind, eid = val
        try:
            if kind == 'entry':
                load_entry_by_id(eid)
            else:
                load_blog_by_id(eid)
            overwrite.value = True  # editing → ready to save back
            print(f"Loaded '{eid}'. Edit fields, then click Add / Update.")
        except Exception as e:
            print('Error:', e)

def on_refresh_load(_):
    load_picker.options = [('— pick to load —', None)] + list_existing_ids()

def on_clear(_):
    overwrite.value = False
    build_form(type_picker.value)  # reset to empty form
    load_picker.value = None
    with load_out:
        clear_output()
        print('Form cleared.')

def delete_entry_by_id(entry_id):
    data = json.loads(ENTRIES_PATH.read_text(encoding='utf-8'))
    before = len(data['entries'])
    data['entries'] = [e for e in data['entries'] if e['id'] != entry_id]
    if len(data['entries']) == before:
        raise ValueError(f"Entry '{entry_id}' not found")
    ENTRIES_PATH.write_text(json.dumps(data, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    # strip from home.json recentNewsIds if present
    if HOME_PATH.exists():
        home = json.loads(HOME_PATH.read_text(encoding='utf-8'))
        ids = [i for i in home.get('recentNewsIds', []) if i != entry_id]
        if ids != home.get('recentNewsIds', []):
            home['recentNewsIds'] = ids
            HOME_PATH.write_text(json.dumps(home, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
            return True
    return False

def delete_blog_by_id(post_id):
    target = None
    for p in BLOG_DIR.glob('*.md'):
        fm = parse_blog_md(p)
        if fm and fm.get('id') == post_id:
            target = p
            break
    if not target:
        raise ValueError(f"Blog '{post_id}' not found")
    target.unlink()
    return target

def on_delete(_):
    with load_out:
        clear_output()
        val = load_picker.value
        if not val:
            print('Pick an entry first.')
            return
        if not delete_confirm.value:
            print("Tick 'Confirm delete' first.")
            return
        kind, eid = val
        try:
            if kind == 'entry':
                home_cleaned = delete_entry_by_id(eid)
                msg = f"Deleted entry '{eid}'."
                if home_cleaned:
                    msg += " Also removed from home.json recentNewsIds."
                print(msg)
            else:
                path = delete_blog_by_id(eid)
                print(f"Deleted blog file {path.name}.")
            delete_confirm.value = False
            load_picker.value = None
            on_refresh_load(None)
            build_form(type_picker.value)  # reset form
            overwrite.value = False
        except Exception as e:
            print('Error:', e)

load_btn.on_click(on_load)
refresh_load_btn.on_click(on_refresh_load)
clear_btn.on_click(on_clear)
delete_btn.on_click(on_delete)

# overwrite checkbox lives in the next cell but we need it here for auto-tick on load.
# Define it now so this cell can reference it; the save cell will display it.
overwrite = widgets.Checkbox(value=False, description='Overwrite if id exists')

display(
    widgets.HTML('<b>Load existing</b> — pick to edit/delete, or skip to create new:'),
    load_picker,
    widgets.HBox([load_btn, refresh_load_btn, clear_btn]),
    widgets.HBox([delete_btn, delete_confirm]),
    load_out,
    widgets.HTML('<hr>'),
    type_picker, form_box, authors_box,
)

HTML(value='<b>Load existing</b> — pick to edit/delete, or skip to create new:')

Dropdown(description='Load existing:', layout=Layout(width='800px'), options=(('— pick to load —', None), ('[p…

Output()

HTML(value='<hr>')

RadioButtons(description='Type:', options=('publication', 'presentation', 'project', 'news', 'blog'), style=De…

### Preview / save

In [ ]:
preview_btn = widgets.Button(description='Preview', button_style='info')
save_btn = widgets.Button(description='Add / Update', button_style='success')
out = widgets.Output()

def split_keywords(s):
    return [t.strip() for t in (s or '').split(',') if t.strip()]

def collect_authors():
    out_authors = []
    for nm, me_cb, _ in author_rows:
        name = (nm.value or '').strip()
        if name:
            out_authors.append({'name': name, 'me': bool(me_cb.value)})
    return out_authors

def collect():
    kind = type_picker.value
    f = {k: w.value for k, w in fields.items()}
    if not f.get('id') or not str(f['id']).strip():
        raise ValueError('id is required')
    if f.get('date') is None:
        raise ValueError('date is required')
    if not f.get('title') or not str(f['title']).strip():
        raise ValueError('title is required')

    if kind == 'blog':
        for k in ('tag', 'excerpt', 'readTime', 'body'):
            if not f.get(k):
                raise ValueError(f'{k} is required for blog')
        return 'blog', {
            'id': f['id'].strip(),
            'title': f['title'].strip(),
            'date': f['date'].isoformat(),
            'tag': f['tag'].strip(),
            'excerpt': f['excerpt'].strip(),
            'readTime': f['readTime'].strip(),
            'body': f['body'].strip(),
        }

    obj = {
        'id': f['id'].strip(),
        'type': kind,
        'date': f['date'].isoformat(),
        'title': f['title'].strip(),
        'keywords': split_keywords(f.get('keywords', '')),
        'description': (f.get('description') or '').strip(),
    }
    if kind == 'publication':
        obj['pubType'] = f['pubType']
        obj['venue'] = (f.get('venue') or '').strip()
        obj['authors'] = collect_authors()
        obj['pdfUrl'] = (f.get('pdfUrl') or '').strip()
    elif kind == 'presentation':
        obj['presType'] = f['presType']
        obj['venue'] = (f.get('venue') or '').strip()
        obj['authors'] = collect_authors()
        obj['attachmentUrl'] = (f.get('attachmentUrl') or '').strip()
    elif kind == 'project':
        obj['mode'] = f['mode']
        obj['authors'] = collect_authors()
        obj['url'] = (f.get('url') or '').strip()
    elif kind == 'news':
        obj['url'] = (f.get('url') or '').strip()
    return 'entry', obj

def save_entry(obj):
    data = json.loads(ENTRIES_PATH.read_text(encoding='utf-8'))
    entries = data['entries']
    idx = next((i for i, e in enumerate(entries) if e['id'] == obj['id']), None)
    if idx is not None:
        if not overwrite.value:
            raise ValueError(f"id '{obj['id']}' exists. Tick 'Overwrite' to replace.")
        entries[idx] = obj
        action = 'updated'
    else:
        entries.append(obj)
        action = 'appended'
    ENTRIES_PATH.write_text(json.dumps({'entries': entries}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return action

def save_blog(obj):
    slug = re.sub(r'[^a-zA-Z0-9-]+', '-', obj['id']).strip('-').lower()
    path = BLOG_DIR / f'{slug}.md'
    if path.exists() and not overwrite.value:
        raise ValueError(f"{path.name} already exists. Tick 'Overwrite' to replace.")
    body = (
        '---\n'
        f"id: {obj['id']}\n"
        f"title: {obj['title']}\n"
        f"date: {obj['date']}\n"
        f"tag: {obj['tag']}\n"
        f"excerpt: {obj['excerpt']}\n"
        f"readTime: {obj['readTime']}\n"
        '---\n\n'
        f"{obj['body']}\n"
    )
    path.write_text(body, encoding='utf-8')
    return path.relative_to(REPO_ROOT)

def on_preview(_):
    with out:
        clear_output()
        try:
            kind, obj = collect()
            print(json.dumps(obj, ensure_ascii=False, indent=2))
        except Exception as e:
            print('Error:', e)

def on_save(_):
    with out:
        clear_output()
        try:
            kind, obj = collect()
            if kind == 'blog':
                path = save_blog(obj)
                print('Wrote', path)
            else:
                action = save_entry(obj)
                print(f"{action} entry '{obj['id']}'")
            # refresh the load dropdown so the new/edited id shows up
            on_refresh_load(None)
        except Exception as e:
            print('Error:', e)

preview_btn.on_click(on_preview)
save_btn.on_click(on_save)
display(widgets.HBox([preview_btn, save_btn, overwrite]), out)

Output()

## 2. Curate Recent News (max 5)

Pick up to 5 entries to feature on the About page. Saved to `content/home.json`.

In [ ]:
def load_entries_for_curate():
    data = json.loads(ENTRIES_PATH.read_text(encoding='utf-8'))
    items = [e for e in data['entries'] if e['type'] in ('publication', 'presentation', 'project', 'news')]
    items.sort(key=lambda e: e.get('date', ''), reverse=True)
    return items

def label_for(entry):
    sub = entry.get('pubType') or entry.get('presType') or entry['type'].title()
    return f"[{sub}] {entry.get('title','')} — {entry.get('date','')}"

all_entries = load_entries_for_curate()
options = [(label_for(e), e['id']) for e in all_entries]
current = json.loads(HOME_PATH.read_text(encoding='utf-8')).get('recentNewsIds', []) if HOME_PATH.exists() else []

curator = widgets.SelectMultiple(
    options=options,
    value=tuple(i for i in current if any(eid == i for _, eid in options)),
    description='Recent News:',
    layout=widgets.Layout(width='800px', height='240px'),
    style={'description_width': '120px'},
)
save_curator_btn = widgets.Button(description='Save selection', button_style='success')
refresh_curator_btn = widgets.Button(description='Refresh list', button_style='')
curator_out = widgets.Output()

def on_save_curator(_):
    with curator_out:
        clear_output()
        ids = list(curator.value)
        if len(ids) > 5:
            print(f'Pick at most 5 (you picked {len(ids)}).')
            return
        HOME_PATH.write_text(json.dumps({'recentNewsIds': ids}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        print(f'Saved {len(ids)} ids to {HOME_PATH.relative_to(REPO_ROOT)}.')

def on_refresh_curator(_):
    items = load_entries_for_curate()
    new_opts = [(label_for(e), e['id']) for e in items]
    curator.options = new_opts
    saved = json.loads(HOME_PATH.read_text(encoding='utf-8')).get('recentNewsIds', []) if HOME_PATH.exists() else []
    curator.value = tuple(i for i in saved if any(eid == i for _, eid in new_opts))

save_curator_btn.on_click(on_save_curator)
refresh_curator_btn.on_click(on_refresh_curator)
display(curator, widgets.HBox([save_curator_btn, refresh_curator_btn]), curator_out)

SelectMultiple(description='Recent News:', index=(6, 2, 3, 0, 7), layout=Layout(height='240px', width='800px')…

Output()

## 3. Build

In [ ]:
build_btn = widgets.Button(description='Build', button_style='info')
build_out = widgets.Output()
build_ok = {'value': False}

def on_build(_):
    with build_out:
        clear_output()
        r = subprocess.run([sys.executable, str(REPO_ROOT / 'scripts' / 'build_content.py')],
                           cwd=REPO_ROOT, text=True, capture_output=True)
        print(r.stdout)
        if r.returncode != 0:
            print(r.stderr); print('Build failed'); build_ok['value'] = False; return
        build_ok['value'] = True
        print('Build OK.')

build_btn.on_click(on_build)
display(build_btn, build_out)

Button(button_style='info', description='Build', style=ButtonStyle())

Output()